In [6]:
import re
from pathlib import Path
import json
import pandas as pd 

def find_metrics(folder):
    sious = [x for x in Path(folder).rglob('*SIoU*.json')]
    return sious

def read_json(json_path):    
    with open(json_path, 'r') as file:
        # Read the content of the file
        content = file.read()
        # Replace single quotes with double quotes
        content = content.replace("'", '"')
        # Load the JSON data
        data = json.loads(content)
        precision = data.get('precision')
        recall = data.get('recall')
        f1_score = data.get('f1_score')
        
    seed, LR, BS = find_seed_lr_bs(json_path.as_posix())        
    
    return {'precision': precision, 'recall': recall, 'f1_score': f1_score, 'seed': seed, 'LR': LR, 'BS': BS}

def read_json2(json_path):    
    with open(json_path, 'r') as file:
        # Read the content of the file
        content = file.read()
        # Replace single quotes with double quotes
        content = content.replace("'", '"')
        # Load the JSON data
        data = json.loads(content)
        precision = data.get('precision')
        recall = data.get('recall')
        f1_score = data.get('f1_score')
        
    seed, LR, BS = json_path.parent.name.split('_'), 0.005, 2        
    
    return {'precision': precision, 'recall': recall, 'f1_score': f1_score, 'seed': seed, 'LR': LR, 'BS': BS}

def find_seed_lr_bs(path):
    """
    Extracts the seed, learning rate (LR), and batch size (BS) from a given file path.

    The function uses a regular expression to match patterns in the file path that represent seed, LR, and BS.
    It is designed to handle various path structures where these values might appear.

    Parameters:
    path (str): The file path from which to extract the seed, LR, and BS.

    Returns:
    tuple or None: A tuple containing the extracted seed (str), LR (str), and BS (str) if found; 
                   otherwise, None if the pattern does not match.
    """
    # Regular expression to extract seed, LR, and BS from the path
    pattern = r"(?P<seed>\d+)(_BS_(?P<BS>\d+))?_LR_(?P<LR>0\.\d+)/"

    # Search for the pattern in the provided path
    match = re.search(pattern, path)
    if match:
        # Extract seed, LR, and BS (if present)
        seed = match.group('seed')
        BS = match.group('BS') if match.group('BS') else None  # Handle optional BS
        LR = match.group('LR')
        return seed, LR, BS
    else:
        print("No match found")
        return None 
    

selector = 'b5_b12'
files = find_metrics(f'/Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/Multi/LR_0.0015/IMG_2048/BANDS__{selector}')
print('Len files:', len(files))

res = [read_json2(x) for x in files]
df = pd.DataFrame(res)
grouped_df = df.groupby(['LR', 'BS']).agg({'precision': ['mean', 'std'], 'recall': ['mean', 'std'], 'f1_score': ['mean', 'std']})
grouped_df.to_pickle(f'metrics_{selector}.pkl')

Len files: 3


In [7]:
grouped_df

precision              recall            f1_score          
              mean       std      mean       std      mean       std
LR    BS                                                            
0.005 2   0.814272  0.008632  0.867061  0.004783  0.839834  0.006838

In [8]:
import os 

x = os.listdir('/Data_large/marine/PythonProjects/MMDET/notebooks/ResCollate')
x = [y for y in x if not 'collater' in y]
x = [y for y in x if not 'sentinel' in y]
x

['metrics_perfect_b3_b4_b7_b11.pkl',
 'metrics_b5_b8.pkl',
 'metrics_b5_b12.pkl',
 'metrics_perfect_b3_b5_b7_b11.pkl',
 'metrics_b5_b10.pkl',
 'metrics_b5_b10_b11.pkl',
 'metrics_perfect_b3_b4_b7.pkl']

In [21]:
import pandas as pd

# Dictionary comprehension to read pickled data into `results`
results = {x: pd.read_pickle(f'/Data_large/marine/PythonProjects/MMDET/notebooks/ResCollate/{x}') for x in x}
results_ref = {}

for key, value in results.items():
    # Check if 'std' column exists, if the first row has 'std' as None, and if there is more than one row
    if key == 'metrics_perfect_b3_b4_b7.pkl':
        # If 'std' is None and there is a second row, use the second row (iloc[1])
        results_ref[key] = value.iloc[1]
    else:
        # Otherwise, use the first row (iloc[0])
        results_ref[key] = value.iloc[0]


In [20]:
results['metrics_perfect_b3_b4_b7.pkl']

precision              recall            f1_score          
               mean       std      mean       std      mean       std
LR     BS                                                            
0.0008 2   0.849916       NaN  0.868639       NaN  0.859175       NaN
0.0009 2   0.851452  0.004319  0.869822  0.003742  0.860538  0.003910
       3   0.854473  0.002101  0.873846  0.002150  0.864050  0.001747
0.001  2   0.855627  0.004456  0.872041  0.004877  0.863755  0.004519
       3   0.853393  0.003026  0.873373  0.003899  0.863266  0.003318
0.002  2   0.854035  0.005697  0.871953  0.007550  0.862897  0.006351
       3   0.852798  0.006080  0.869822  0.004807  0.861225  0.005417
0.003  2   0.813313  0.087237  0.841183  0.066848  0.826866  0.077665
       3   0.854281  0.003155  0.873136  0.004048  0.863602  0.003037
0.004  2   0.580085  0.318711  0.722189  0.173561  0.630111  0.270254
       3   0.684780  0.245218  0.773964  0.143932  0.723866  0.202236
0.005  2   0.835528  0.024884  0.860947  0.022594  0.848046  0.023779
       3   0.384368  0.373869  0.490888  0.351493  0.420277  0.369208

In [22]:
results_ref

{'metrics_perfect_b3_b4_b7_b11.pkl': precision  mean    0.841515
            std     0.005173
 recall     mean    0.872426
            std     0.003063
 f1_score   mean    0.856689
            std     0.003926
 Name: (0.0009, 2), dtype: float64,
 'metrics_b5_b8.pkl': precision  mean    0.804649
            std     0.005499
 recall     mean    0.863116
            std     0.003615
 f1_score   mean    0.832856
            std     0.004512
 Name: (0.005, 2), dtype: float64,
 'metrics_b5_b12.pkl': precision  mean    0.814272
            std     0.008632
 recall     mean    0.867061
            std     0.004783
 f1_score   mean    0.839834
            std     0.006838
 Name: (0.005, 2), dtype: float64,
 'metrics_perfect_b3_b5_b7_b11.pkl': precision  mean    0.845372
            std     0.003836
 recall     mean    0.875740
            std     0.001871
 f1_score   mean    0.860285
            std     0.002577
 Name: (0.0009, 2), dtype: float64,
 'metrics_b5_b10.pkl': precision  mean    0.809